# <strong>An Introduction to Dwarf</strong>

In this exercise, we review different types of compiled binaries and the structure of debugging information in binary programs, and we will learn how debuggers use this information. 

## <strong>Required Reading</strong>
### <strong>What is a binary file?</strong>

A binary file is a compiled file from a source code that contains machine code. A binary file has different types, such as executable, object file, etc. In the next section, we discuss different types of binary files.

### <strong>Binary Types</strong>

Binary program files have different types. The type of binary file is stored in the header section of the binary. This section introduces three different types of binary files, including object files, executable, and dynamic shared library.

An object file is a compiled file that contains machine code. For instance, imagine that a program source code consists of two source files, such as file1.c and file2.c. The compiler first compiles each file separately and creates two files, file1.o and file2.o, which are called object files or relocatable files. An object file is not executable. This type of binary has a signature "ET_REL" in the binary header. In the later part of the compilation phase, the linker links these two files and any system libraries to make an executable file. After the linking phase, the output is an executable file, and we can run it. The type of output binary of this phase is executable type. This binary type has a signature "ET_EXEC" in the binary header. Another binary type is dynamic shared library which has a signature "ET_DYN" in the header file. The only difference between a dynamic shared library and an executable is that the library does not have an entry point, while an executable has. The entry point is a location in memory that defines where a compiled program starts to be executed (e.g., main function in the program).

The following code shows how to detect the type of binary. In order to run the code, we first need to install pyelftools, which is a parser for elf binaries. In your experiment pyelftools are already installed. Below you can see analysis code we will use. You can also access this code on your `analysis` node, in `/tmp/analysis directory`. Change the 'path to binary' to point to binary1.out on your disk (also in `/tmp/analysis` directory). 
```
from elftools.elf.elffile import ELFFile

filename = 'path to binary'
with open(filename, 'rb') as file:
	elffile = ELFFile(file)
	elf_type = elffile.header.e_type
	print(elf_type)
```
When you type: `python3 script1.py` you should see "ET_DYN", which indicates that `binary1.out` is a dynamic shared library. 

### <strong>Debug Sections in Binary</strong>

When we compile a binary from the source code, we can compile it with the debug information (e.g., by supplying -g flag to gcc or g++). If debug information is present, you are able to extract more information about binary, such as mapping between source code line and memory offsets. In this exercise, we are focusing on ELF executable file format, which is an executable format in Linux binaries. The debugging information in Linux binaries is called "dwarf". the dwarf has 12 different sections, but we only focus on two debug sections, including debug info and debug_line sections.

If you compile your binary with the -g command option, the binary will be compiled with debug information. If you want to analyze whether your binary file contains debug information or not, you can use the following command: `file binary_file`

If the binary file is compiled with debug information, you will see this result: `"with debug_info, not stripped"`

For example, you will see this for binary1.out.

Now, we will dive into the details of two debug sections, debug_info, and debug_line section. Debug_info section contains global variables, functions, and its corresponding local variables information. Debug_line section contains the information about mapping between source code lines and memory addresses. 

#### <strong>Debug_Info Section</strong>

Debug_info is a tree structure in a binary. The main component of this structure is called Debugging Information Entry or DIE. Each DIE has a tag, type, and set of attributes. You can use different tools to print out the dwarf info of a binary. One of these tools is objdump: `objdump --dwarf=info binary_file`

The first DIE in the output of this command is the "compilation unit". As we previously mentioned, a source code can be split into different files. When the linker links the object files (created from these source files) to create an executable file, we have a unit in the binary level, corresponding to each source file, which is called the compilation unit. In the debug info section, each compilation unit has a DIE with the tag "DW_TAG_compile_unit". In the tree structure of a DIEs, the compilation unit is a root. If you look at the output of the command above for the binary binary1.out, you will see that binary1.out has two compilation units.

Each source file may have several functions. In the tree structure of the debug info section, each function is a child of a compilation unit in which it was defined. In this structure, each function is defined with a tag DW_TAG_subprogram. In the picture below, you can see the structure of a DIE corresponding to a function.

Each DIE has some attributes. The DW_AT_name can define the function name. In the example below, "badsink" is the name of the function. Each function can have some parameters and local variables. In the debug info structure, function arguments and local variables are defined with "DW_TAG_formal_parameter" and "DW_TAG_variable" tags, respectively. 

<img src="resources/dwarf/DIE.png">

Now we give a short script to extract the name of functions in the dwarf info section. You can access the code on `analysis` node in `/tmp/analysis` folder. The code is `script2.py`.

This code will print the name of the functions in `binary1.out`, which have a "DW_TAG_subprogram" tag.

This code prints the name of a function. If in the main function you print the type of die you will see it is an object of class 'elftools.dwarf.die.DIE'. If in <a href="https://github.com/eliben/pyelftools/blob/master/elftools/dwarf/die.py">pyelftools repository</a> you look at DIE class attributes, you will see that DIE has an attribute "tag" which defines the tag of a DIE and attributes. Then if you ask for the value of the attribute ('DW_AT_name') you can see the name of the function.

Running this code with binary1.out should produce the following output: 
```
  python3 script2.py binary1.out
  
  Found a compile unit at offset 0, length 1045
  Found a compile unit at offset 1049, length 1058
CWE121_Stack_Based_Buffer_Overflow__CWE129_rand_51b_badSink
main
  CWE121_Stack_Based_Buffer_Overflow__CWE129_rand_51_bad
```

#### <strong>Debug_line Section</strong>

One of the other sections in debug information is the `debug_line` section. This section contains the mapping between a source code line and memory offset in the binary code. In other words, it contains information about which assembly instructions correspond to which source code lines. The debug_line section contains all the compilation units, and for each compilation unit, it contains the mapping between a source code line and memory offsets. We first analyze this information using `objdump`. If you run this command in the terminal for `binary1.out`, you will see the output shown below.

```
objdump --dwarf=decodedline binary1.out

binary1.out:     file format elf64-x86-64

Contents of the .debug_line section:

CU: ./CWE121_Stack_Based_Buffer_Overflow__CWE129_rand_51b.c:
File name                            Line number    Starting address    View
_Buffer_Overflow__CWE129_rand_51b.c           23              0x13a9
_Buffer_Overflow__CWE129_rand_51b.c           23              0x13b8
_Buffer_Overflow__CWE129_rand_51b.c           26              0x13c7
_Buffer_Overflow__CWE129_rand_51b.c           29              0x13ef
_Buffer_Overflow__CWE129_rand_51b.c           31              0x13f5
_Buffer_Overflow__CWE129_rand_51b.c           33              0x1402
_Buffer_Overflow__CWE129_rand_51b.c           33              0x1409
_Buffer_Overflow__CWE129_rand_51b.c           35              0x140b
_Buffer_Overflow__CWE129_rand_51b.c           33              0x141b
_Buffer_Overflow__CWE129_rand_51b.c           33              0x141f
_Buffer_Overflow__CWE129_rand_51b.c           43              0x1425
_Buffer_Overflow__CWE129_rand_51b.c           40              0x1427
_Buffer_Overflow__CWE129_rand_51b.c           43              0x1433
_Buffer_Overflow__CWE129_rand_51b.c           43              0x144a


CU: ./CWE121_Stack_Based_Buffer_Overflow__CWE129_rand_51a.c:
File name                            Line number    Starting address    View
_Buffer_Overflow__CWE129_rand_51a.c           26              0x144a
_Buffer_Overflow__CWE129_rand_51a.c           29              0x1457
_Buffer_Overflow__CWE129_rand_51a.c           31              0x145e
_Buffer_Overflow__CWE129_rand_51a.c           31              0x146a
_Buffer_Overflow__CWE129_rand_51a.c           31              0x1487
_Buffer_Overflow__CWE129_rand_51a.c           31              0x14a4
_Buffer_Overflow__CWE129_rand_51a.c           32              0x14a7
_Buffer_Overflow__CWE129_rand_51a.c           33              0x14b1
_Buffer_Overflow__CWE129_rand_51a.c           82              0x14b9
_Buffer_Overflow__CWE129_rand_51a.c           84              0x14cc
_Buffer_Overflow__CWE129_rand_51a.c           84              0x14d6
_Buffer_Overflow__CWE129_rand_51a.c           91              0x14dd
_Buffer_Overflow__CWE129_rand_51a.c           92              0x14e9
_Buffer_Overflow__CWE129_rand_51a.c           93              0x14f3
_Buffer_Overflow__CWE129_rand_51a.c           95              0x14ff
_Buffer_Overflow__CWE129_rand_51a.c           96              0x1504
_Buffer_Overflow__CWE129_rand_51a.c           96              0x1506
```

Since a binary may contain different compilation units, you should first determine which source line number belongs to which function. Once you have the function name, you can easily extract the compilation unit of this function from a DIE in the dwarf info section, and then you can explore its memory locations.

The code below extracts the mapping between source code and memory offsets for each compilation unit. You can access the code on your `analysis` node in `/tmp/analysis/` folder. The code is `script3.py`.

```
	import argparse
	from elftools.elf.elffile import ELFFile
	
	def debug_line(binary_file):
		lines_offsets = {}
		with open(binary_file, 'rb') as file:
			elffile = ELFFile(file)
			if not elffile.has_dwarf_info():
				print('  file has no DWARF info')
				return
			dwarfinfo = elffile.get_dwarf_info()
			for CU in dwarfinfo.iter_CUs():
				lines_program = []
				cu_die = CU.get_top_DIE()
				cu_name = cu_die.attributes['DW_AT_name'].value.decode()
				lines = dwarfinfo.line_program_for_CU(CU)
				debugsec_lines = lines.get_entries()
				for line in debugsec_lines:
					#print(line)
					if line.state is not None:
						lines_program.append((line.state.line,hex(line.state.address)))
				lines_offsets[cu_name] = lines_program
	
		return lines_offsets
	
	if __name__ == '__main__':
		parser = argparse.ArgumentParser()
		parser.add_argument('binary', help='path to binary folder')
		parser.add_argument('line_number', help='cu')
		args = parser.parse_args()
		binary_file = args.binary
		lines_offsets = debug_line(binary_file)
		print(lines_offsets)
```

When you run it you should see the output like below: 
```
    python3 script3.py binary1.out
    
    {'CWE121_Stack_Based_Buffer_Overflow__CWE129_rand_51b.c': [(23, '0x13a9'), (23, '0x13b8'), (26, '0x13c7'), (29, '0x13ef'), (31, '0x13f5'), (33, '0x1402'), (33, '0x1409'), (35, '0x140b'), (33, '0x141b'), (33, '0x141f'), (43, '0x1425'), (40, '0x1427'), (43, '0x1433'), (43, '0x144a')], 'CWE121_Stack_Based_Buffer_Overflow__CWE129_rand_51a.c': [(26, '0x144a'), (29, '0x1457'), (31, '0x145e'), (31, '0x146a'), (31, '0x1487'), (31, '0x14a4'), (32, '0x14a7'), (33, '0x14b1'), (82, '0x14b9'), (84, '0x14cc'), (84, '0x14d6'), (91, '0x14dd'), (92, '0x14e9'), (93, '0x14f3'), (95, '0x14ff'), (96, '0x1504'), (96, '0x1506')]}
```

### Step 0: Starting the Lab

Click the button to begin creating the experiment.

<strong>Note:</strong> If your buttons are not displaying, click on the <img width='20px' height='20px' style='margin-left: 1px;' src='resources/fast_forward.png'> icon at the top of your notebook to render all widgets.

In [1]:
# Click on the button below to start your lab.
from IPython.display import display, HTML
import sys
import os
import subprocess
import re
import threading
import queue
import time
import logging
import shutil
import tempfile
import tarfile
from pathlib import Path
import ipywidgets as widgets

# The lab name.
labname = "dwarf"

# Adding the "resources/" directory so that we can import the start.py file.
module_dir = os.path.join(os.getcwd(), 'resources')
if module_dir not in sys.path:
    sys.path.append(module_dir)

# Importing the prepare_lab function.
from functions import *

# Required for Step 5 to work.
step1Complete = step2Complete = step3Complete = runAllSteps = False

def setup_lab():
    try:
        # Check if ipywidgets is imported.
        if 'widgets' not in globals():
            raise ImportError("Jupyter Widgets not imported correctly.")
    except ImportError as e:
        display(HTML(
            f"<div style='color: red;'>"
            f"Jupyter Widgets was not imported correctly. Error: {e}<br>"
            "Please re-run <code>install_notebooks.sh</code>, <u>refresh your browser tab</u>, then try again."
            "</div>"
        ))
        raise e

    # Defining UI.
    output0 = widgets.Output()
    startButton = widgets.Button(description="Start Lab")
    

    # Defining the button handler.
    def on_start_clicked(b):
        prepare_lab(labname, output0)

    startButton.on_click(on_start_clicked)
    display(startButton, output0)

setup_lab()

Button(description='Start Lab', style=ButtonStyle())

Output()

<hr>

If you previously stopped your lab by using the "Stop Lab" button at the bottom of the notebook, you may restore your progress below by clicking "Load Lab". <u>You do not have to load your lab if you signed out, closed your notebook, or exited your node(s) or XDC by using ```exit```.</u>

In [2]:
# Click the button below to load your lab.
def loadlab(b):
    load_lab(labname, output0_2)

# Creating the button.
loadButton = widgets.Button(description="Load Lab")

# Creating an output area.
output0_2 = widgets.Output()

# Run the command on click.
loadButton.on_click(loadlab)

# Display the output.
display(loadButton, output0_2)

Button(description='Load Lab', style=ButtonStyle())

Output()

### Step 1: Examining `binary2.out`

In this exercise, you will practice everything you have read above. For this assignment, we will use the file `binary2.out` from `/tmp/analysis` on your `analysis` node.

The first three questions are open-ended, and your answers will be saved at each step. The final step of this lab will generate a submission with your answers included.

<strong>Your task:</strong> Run `script1.py` for `binary2.out`. What is the type of this binary? Explain what this means.

In [3]:
# Click the button below to check your work.
def step_1():
    # Important variables that must be accessed outside of this function.
    global step1Complete, result

    # Replace backticks with single quotes in the input.
    safe_value = userInput1.value.replace("`", "'")
    
    # Build the SSH command to write to the file using 'cat' reading from stdin.
    ssh_command = [
        "ssh",
        "-i", "/home/USERNAME_GOES_HERE/.ssh/merge_key",
        "USERNAME_GOES_HERE@analysis",
        "cat > /home/.checker/responses/step_1_answer.txt"
    ]
    
    # Pass safe_value as input so that no shell quoting is needed.
    result = subprocess.run(ssh_command, input=safe_value, text=True)

    if result.returncode == 255:
        output1.clear_output()
        with output1:
            display(HTML("<span style='color: red;'>You have not started this lab yet. Please click \"Start Lab\" at the top of this notebook.</span>"))
            return
    
    elif result.returncode == 1:
        output1.clear_output()
        with output1:
            display(HTML("<span style='color: red;'>There was an error saving your response.</span>"))
            step1Complete = False
            
    elif result.returncode == 0:
        output1.clear_output()
        with output1:
            display(HTML("<span style='color: green;'>Your response was saved.</span>"))
            step1Complete = True

def check_step_1(b):
    if warn_student(labname):
        output1.clear_output()
        with output1:
            display(HTML("<span style='color: red;'><strong>WARNING:</strong> You have an autosaved lab that you have not yet loaded. If you would like to load your progress, click \"Load Lab\" at the top of the notebook. Otherwise, clicking on this button again will assume you're restarting the lab!</span>"))
    else:
        step_1()
        if not runAllSteps:
            safe_value = userInput1.value.replace("`", "'")
            user_input_quoted = shlex.quote(safe_value)
            trigger_save(labname, "1", 1 if result.returncode == 0 else 0, user_input_quoted)

# Retrieve the student's response. First, create a loading spinner, since this could take a second or two.
loading1 = widgets.Output()
display(loading1)
with loading1:
    loading1.clear_output()
    display(HTML("<span>Loading your saved response... <img width='12px' height='12px' style='margin-left: 3px;' src='resources/loading.gif'></span>"))

# Creating a text area. We will need to assign output of process to the value of this input.
userInput1 = widgets.Textarea(
    placeholder='Type your response here',
    description='Response:',
    layout=widgets.Layout(width='75%', height='150px', margin='10px')
)

def on_input_change(change):
    # Replace backticks with single quotes in the new value.
    new_val = change['new'].replace("`", "'")
    # Only update if there's a change (to avoid unnecessary recursion).
    if new_val != change['new']:
        userInput1.value = new_val

userInput1.observe(on_input_change, names='value')

# Checking if the step has been answered.
result = subprocess.run('ssh -o StrictHostKeyChecking=no -i /home/USERNAME_GOES_HERE/.ssh/merge_key USERNAME_GOES_HERE@analysis "cat /home/.checker/responses/step_1_answer.txt 2> /dev/null"', capture_output=True, text=True, shell=True)
userInput1.value = result.stdout

# After the student's response was loaded, clear the output.
loading1.clear_output()

# Creating the button.
button = widgets.Button(description="Save Response")

# Creating an output area.
output1 = widgets.Output()

# Run the command on click.
button.on_click(check_step_1)

# Display the output.
display(userInput1, button, output1)

Output()

Textarea(value='asdf', description='Response:', layout=Layout(height='150px', margin='10px', width='75%'), pla…

Button(description='Save Response', style=ButtonStyle())

Output()

### Step 2: Debug Symbols

<strong>Answer this question:</strong> Check if the binary file is compiled with debug symbols. Explain how you found that out? 

In [4]:
# Click the button below to check your work.
def step_2():
    # Important variables that must be accessed outside of this function.
    global step2Complete, result

    # Replace backticks with single quotes in the input.
    safe_value = userInput2.value.replace("`", "'")
    
    # Build the SSH command to write to the file using 'cat' reading from stdin.
    ssh_command = [
        "ssh",
        "-i", "/home/USERNAME_GOES_HERE/.ssh/merge_key",
        "USERNAME_GOES_HERE@analysis",
        "cat > /home/.checker/responses/step_2_answer.txt"
    ]
    
    # Pass safe_value as input so that no shell quoting is needed.
    result = subprocess.run(ssh_command, input=safe_value, text=True)

    if result.returncode == 255:
        output2.clear_output()
        with output2:
            display(HTML("<span style='color: red;'>You have not started this lab yet. Please click \"Start Lab\" at the top of this notebook.</span>"))
            return
    
    elif result.returncode == 1:
        output2.clear_output()
        with output2:
            display(HTML("<span style='color: red;'>There was an error saving your response.</span>"))
            step2Complete = False
            
    elif result.returncode == 0:
        output2.clear_output()
        with output2:
            display(HTML("<span style='color: green;'>Your response was saved.</span>"))
            step2Complete = True

def check_step_2(b):
    if warn_student(labname):
        output2.clear_output()
        with output2:
            display(HTML("<span style='color: red;'><strong>WARNING:</strong> You have an autosaved lab that you have not yet loaded. If you would like to load your progress, click \"Load Lab\" at the top of the notebook. Otherwise, clicking on this button again will assume you're restarting the lab!</span>"))
    else:
        step_2()
        if not runAllSteps:
            safe_value = userInput2.value.replace("`", "'")
            user_input_quoted = shlex.quote(safe_value)
            trigger_save(labname, "2", 1 if result.returncode == 0 else 0, user_input_quoted)

# Retrieve the student's response. First, create a loading spinner, since this could take a second or two.
loading2 = widgets.Output()
display(loading2)
with loading2:
    loading2.clear_output()
    display(HTML("<span>Loading your saved response... <img width='12px' height='12px' style='margin-left: 3px;' src='resources/loading.gif'></span>"))

# Creating a text area. We will need to assign output of process to the value of this input.
userInput2 = widgets.Textarea(
    placeholder='Type your response here',
    description='Response:',
    layout=widgets.Layout(width='75%', height='150px', margin='10px')
)

def on_input_change(change):
    # Replace backticks with single quotes in the new value.
    new_val = change['new'].replace("`", "'")
    # Only update if there's a change (to avoid unnecessary recursion).
    if new_val != change['new']:
        userInput2.value = new_val

userInput2.observe(on_input_change, names='value')

# Checking if the step has been answered.
result = subprocess.run('ssh -o StrictHostKeyChecking=no -i /home/USERNAME_GOES_HERE/.ssh/merge_key USERNAME_GOES_HERE@analysis "cat /home/.checker/responses/step_2_answer.txt 2> /dev/null"', capture_output=True, text=True, shell=True)
userInput2.value = result.stdout

# After the student's response was loaded, clear the output.
loading2.clear_output()

# Creating the button.
button = widgets.Button(description="Save Response")

# Creating an output area.
output2 = widgets.Output()

# Run the command on click.
button.on_click(check_step_2)

# Display the output.
display(userInput2, button, output2)

Output()

Textarea(value='test', description='Response:', layout=Layout(height='150px', margin='10px', width='75%'), pla…

Button(description='Save Response', style=ButtonStyle())

Output()

### Step 3: Compilation Units

<strong>Answer this question:</strong> Find out how many compilation units this binary has? Print the name of each.

In [5]:
# Click the button below to check your work.
def step_3():
    # Important variables that must be accessed outside of this function.
    global step3Complete, result

    # Replace backticks with single quotes in the input.
    safe_value = userInput3.value.replace("`", "'")
    
    # Build the SSH command to write to the file using 'cat' reading from stdin.
    ssh_command = [
        "ssh",
        "-i", "/home/USERNAME_GOES_HERE/.ssh/merge_key",
        "USERNAME_GOES_HERE@analysis",
        "cat > /home/.checker/responses/step_3_answer.txt"
    ]
    
    # Pass safe_value as input so that no shell quoting is needed.
    result = subprocess.run(ssh_command, input=safe_value, text=True)

    if result.returncode == 255:
        output3.clear_output()
        with output3:
            display(HTML("<span style='color: red;'>You have not started this lab yet. Please click \"Start Lab\" at the top of this notebook.</span>"))
            return
    
    elif result.returncode == 1:
        output3.clear_output()
        with output3:
            display(HTML("<span style='color: red;'>There was an error saving your response.</span>"))
            step3Complete = False
            
    elif result.returncode == 0:
        output3.clear_output()
        with output3:
            display(HTML("<span style='color: green;'>Your response was saved.</span>"))
            step3Complete = True

def check_step_3(b):
    if warn_student(labname):
        output3.clear_output()
        with output3:
            display(HTML("<span style='color: red;'><strong>WARNING:</strong> You have an autosaved lab that you have not yet loaded. If you would like to load your progress, click \"Load Lab\" at the top of the notebook. Otherwise, clicking on this button again will assume you're restarting the lab!</span>"))
    else:
        step_3()
        if not runAllSteps:
            safe_value = userInput3.value.replace("`", "'")
            user_input_quoted = shlex.quote(safe_value)
            trigger_save(labname, "3", 1 if result.returncode == 0 else 0, user_input_quoted)

# Retrieve the student's response. First, create a loading spinner, since this could take a second or two.
loading3 = widgets.Output()
display(loading3)
with loading3:
    loading3.clear_output()
    display(HTML("<span>Loading your saved response... <img width='12px' height='12px' style='margin-left: 3px;' src='resources/loading.gif'></span>"))

# Creating a text area. We will need to assign output of process to the value of this input.
userInput3 = widgets.Textarea(
    placeholder='Type your response here',
    description='Response:',
    layout=widgets.Layout(width='75%', height='150px', margin='10px')
)

def on_input_change(change):
    # Replace backticks with single quotes in the new value.
    new_val = change['new'].replace("`", "'")
    # Only update if there's a change (to avoid unnecessary recursion).
    if new_val != change['new']:
        userInput3.value = new_val

userInput3.observe(on_input_change, names='value')

# Checking if the step has been answered.
result = subprocess.run('ssh -o StrictHostKeyChecking=no -i /home/USERNAME_GOES_HERE/.ssh/merge_key USERNAME_GOES_HERE@analysis "cat /home/.checker/responses/step_3_answer.txt 2> /dev/null"', capture_output=True, text=True, shell=True)
userInput3.value = result.stdout

# After the student's response was loaded, clear the output.
loading3.clear_output()

# Creating the button.
button = widgets.Button(description="Check Work")

# Creating an output area.
output3 = widgets.Output()

# Run the command on click.
button.on_click(check_step_3)

# Display the output.
display(userInput3, button, output3)

Output()

Textarea(value='asdf', description='Response:', layout=Layout(height='150px', margin='10px', width='75%'), pla…

Button(description='Check Work', style=ButtonStyle())

Output()

### Step 4: Functions and Compilation Units

<strong>Your task:</strong> Write a script to print the name of functions for each compilation unit and the name of arguments and local variables for each function. Run it and view the output. 

Start by creating a copy of `script2.py` and modify it. Name your copy as `script2_modified.py`. Ensure that it remains in the same directory as `script2.py`. <strong>The notebook will find this file and make sure it runs without errors.</strong> The output of this file and a copy of `script2_modified.py` will be saved with your submission.

Click the "Save Script" button to save your work.

In [11]:
# Click the button below to check your work.
step4Complete = False

def step_4():
    # Important variables that must be accessed outside of this function.
    global step4Complete, result

    # Loading, in case the check is slow.
    with output4:
        output4.clear_output()
        display(HTML("<span><img width='12px' height='12px' style='margin-left: 3px;' src='resources/loading.gif'></span>"))
    
    result = subprocess.run([
        "ssh",
        "-i", "/home/USERNAME_GOES_HERE/.ssh/merge_key",
        "USERNAME_GOES_HERE@analysis",
        "/home/.checker/step_4.py"
    ])

    # Success.
    if (result.returncode == 0):
        output4.clear_output()
        with output4:
            display(HTML("<span style='color: green;'><code>script2_modified.py</code> was found. The file and its output has been saved with your submission.</span>"))
            step4Complete = True

    # File has not been found.
    elif (result.returncode == 1):
        output4.clear_output()
        with output4:
            display(HTML("<span style='color: red;'><code>script2_modified.py</code> wasn't found. Ensure it stays in the same directory as <code>script3.py</code>.</span>"))
            step4Complete = False

    # File returns an error.
    elif (result.returncode == 2):
        output4.clear_output()
        with output4:
            display(HTML("<span style='color: red;'>Your <code>script2_modified.py</code> was found, but produces an error. Please fix any bugs and try again. (Hint: Does it return code 0?)</span>"))
            step4Complete = False

    # Checker script isn't ran correctly. Shouldn't happen.
    elif (result.returncode == 3):
        output4.clear_output()
        with output4:
            display(HTML("<span style='color: red;'>There was an error running this step. Please contact your instructor/TA.</span>"))
            step4Complete = False
    

def check_step_4(b):
    if (warn_student(labname)):
        output4.clear_output()
        with output4:
            display(HTML("<span style='color: red;'><strong>WARNING:</strong> You have an autosaved lab that you have not yet loaded. If you would like to load your progress, click \"Load Lab\" at the top of the notebook. Otherwise, clicking on this button again will assume you're restarting the lab!</span>"))
    else:
        step_4()

        # Auto-save.
        if (not runAllSteps):
            trigger_save(labname, "4", result.returncode)

# Creating the button.
button = widgets.Button(description="Test File")

# Creating an output area.
output4 = widgets.Output()

# Run the command on click.
button.on_click(check_step_4)

# Display the output.
display(button, output4)

Button(description='Test File', style=ButtonStyle())

Output()

### Step 5: Memory Offsets

<strong>Your task:</strong> Write a script to extract memory offsets corresponding to line number 33 for each compilation unit. Run it and show the output.

Start by creating a copy of `script3.py` and modify it. Name your copy as `script3_modified.py`. Ensure that it remains in the same directory as `script3.py`. <strong>The notebook will find this file and make sure it runs without errors.</strong> The output of this file and a copy of `script3_modified.py` will be saved with your submission.

In [7]:
# Click the button below to check your work.
step5Complete = False

def step_5():
    # Important variables that must be accessed outside of this function.
    global step5Complete, result

    # Loading, in case the check is slow.
    with output5:
        output5.clear_output()
        display(HTML("<span><img width='12px' height='12px' style='margin-left: 3px;' src='resources/loading.gif'></span>"))
    
    result = subprocess.run([
        "ssh",
        "-i", "/home/USERNAME_GOES_HERE/.ssh/merge_key",
        "USERNAME_GOES_HERE@analysis",
        "/home/.checker/step_5.py"
    ])

    # Success.
    if (result.returncode == 0):
        output5.clear_output()
        with output5:
            display(HTML("<span style='color: green;'><code>script3_modified.py</code> was found. The file and its output has been saved with your submission.</span>"))
            step5Complete = True

    # File has not been found.
    elif (result.returncode == 1):
        output5.clear_output()
        with output5:
            display(HTML("<span style='color: red;'><code>script3_modified.py</code> wasn't found. Ensure it stays in the same directory as <code>script3.py</code>.</span>"))
            step5Complete = False

    # File returns an error.
    elif (result.returncode == 2):
        output5.clear_output()
        with output5:
            display(HTML("<span style='color: red;'>Your <code>script3_modified.py</code> was found, but produces an error. Please fix any bugs and try again. (Hint: Does it return code 0?)</span>"))
            step5Complete = False
    

def check_step_5(b):
    if (warn_student(labname)):
        output5.clear_output()
        with output5:
            display(HTML("<span style='color: red;'><strong>WARNING:</strong> You have an autosaved lab that you have not yet loaded. If you would like to load your progress, click \"Load Lab\" at the top of the notebook. Otherwise, clicking on this button again will assume you're restarting the lab!</span>"))
    else:
        step_5()

        # Auto-save.
        if (not runAllSteps):
            trigger_save(labname, "5", result.returncode)

# Creating the button.
button = widgets.Button(description="Test File")

# Creating an output area.
output5 = widgets.Output()

# Run the command on click.
button.on_click(check_step_5)

# Display the output.
display(button, output5)

Button(description='Test File', style=ButtonStyle())

Output()

### Step 6: Generate Your Submission

<strong>Your task:</strong> Once you have completed all previous tasks, click "Generate Submission" below to create a submission. You will turn this into your instructor.

This button will be locked until Steps 1-5 are completed.

In [8]:
# Click the button below to check your work.
def step_6():
    global step1Complete, step2Complete, step3Complete, step4Complete, step5Complete, result

    if (not step1Complete or not step2Complete or not step3Complete or not step4Complete or not step5Complete):
        output6.clear_output()
        with output6:
            display(HTML("<span style='color: red;'>Please complete all previous before generating your submission.</span>"))
            step6Complete = False
            return

    with output6:
        output6.clear_output()
        display(HTML("<span>Generating your final submissions. Please wait a few seconds... <img width='12px' height='12px' style='margin-left: 2px;' src='resources/loading.gif'></span>"))

    home = Path.home()
    temp_dir = tempfile.mkdtemp()
    
    # Define target files and their destination in temp_dir.
    file_paths = [
        ("node-0:/tmp/node-0/paws/client.c", os.path.join(temp_dir, "client_node0.c")),
        ("node-1:/tmp/node-1/paws/client.c", os.path.join(temp_dir, "client_node1.c"))
    ]

    # Copy files from remote nodes using scp.
    for remote, local in file_paths:
        result = subprocess.run(
            ['scp', '-o', 'StrictHostKeyChecking=no', remote, local],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL
        )

    # Check for optional report files in the user's home directory.
    docx_path = home / "USERNAME_GOES_HERE_report.docx"
    pdf_path = home / "USERNAME_GOES_HERE_report.pdf"
    doc_found = False

    if docx_path.exists():
        shutil.copy(docx_path, temp_dir)
        doc_found = True
    elif pdf_path.exists():
        shutil.copy(pdf_path, temp_dir)
        doc_found = True

    # Create tarball entirely within the temp directory.
    tar_name = "USERNAME_GOES_HERE_worm.tar.gz"
    tar_path = os.path.join(temp_dir, tar_name)
    with tarfile.open(tar_path, "w:gz") as tar:
        for file_name in os.listdir(temp_dir):
            full_path = os.path.join(temp_dir, file_name)
            # Don't add the tarball itself into the tar
            if full_path != tar_path:
                tar.add(full_path, arcname=file_name)

    # Move the tarball to the home directory.
    final_tar_path = home / tar_name
    shutil.move(tar_path, final_tar_path)

    # Output messages based on step completion.
    if doc_found:
        output5.clear_output()
        with output5:
            display(HTML("<span style='color: green;'>Your submission (USERNAME_GOES_HERE_worm.tar.gz) was generated WITH your report. It is located in the sidebar of your XDC.</span>"))
            step5Complete = True

    else:
        output5.clear_output()
        with output5:
            display(HTML("<span style='color: green;'>Your submission (USERNAME_GOES_HERE_worm.tar.gz) was generated WITHOUT your report. It is located in the sidebar of your XDC. Ensure that you include your report in your final submission!</span>"))
            step5Complete = True

def check_step_5(b):
    if (warn_student(labname)):
        output5.clear_output()
        with output5:
            display(HTML("<span style='color: red;'><strong>WARNING:</strong> You have an autosaved lab that you have not yet loaded. If you would like to load your progress, click \"Load Lab\" at the top of the notebook. Otherwise, clicking on this button again will assume you're restarting the lab!</span>"))
    else:
        step_5()

        # Auto-save.
        if (not runAllSteps):
            if ("result" in locals()):
                trigger_save(labname, "5", "0")

            else:
                trigger_save(labname, "5", "1")

# Creating the button.
button = widgets.Button(description="Check Work")

# Creating an output area.
output5 = widgets.Output()

# Run the command on click.
button.on_click(check_step_5)

# Display the output.
display(button, output5)

Button(description='Check Work', style=ButtonStyle())

Output()

## <strong>Grading</strong>

To check your overall grade, click on the button below.

In [9]:
# Click the button below to check your overall grade.
steps_to_check = [step_1, step_2, step_3, step_5]   

# Function to calculate grade after refreshing the cell.
def calculate_grade(b):
    # To not auto-save at each step.
    global runAllSteps
    runAllSteps = True

    with gradeOutput:
        gradeOutput.clear_output()
        display(HTML("<span>Testing all steps. Please wait.</span> \
            <span><img width='12px' height='12px' style='margin-left: 3px;' src='resources/loading.gif'></span>"))
    
    # Required for checking the boolean values.
    for func in steps_to_check:
        func()

    steps = [step1Complete, step2Complete, step3Complete, step5Complete]
    output = ""
    stepsCorrect = 0

    # Adding one because Step 4 is not graded, and is not counted for within "steps".
    numOfSteps = len(steps) + 1

    for i in range(1, numOfSteps):
        if i == 4:
            output += "<div style='color: orange;'>Step 4 is not automatically graded.</div>"
            
        if steps[i - 1]:
            stepsCorrect += 1
            # Hardcoding this. Will be more difficult to grade if future steps are added.
            if (i == 4):
                output += "<div style='color: green;'>Step 5 is complete.</div>"

            else:
                output += "<div style='color: green;'>Step " + str(i) + " is complete.</div>"

        else:
            # Hardcoding this. Will be more difficult to grade if future steps are added.
            if (i == 4):
                output += "<div style='color: red;'>Step 5 is incomplete.</div>"

            else:
                output += "<div style='color: red;'>Step " + str(i) + " is incomplete.</div>"
                
    # Removing the extra step that was added (Step 4) since it should not be counted with the overall grade.
    output += "<div style='color: black;'>You have " + str(stepsCorrect) + " out of " + str(numOfSteps - 1) + " steps completed.</div>"

    with gradeOutput:
        gradeOutput.clear_output()
        display(HTML(output))

    # Disable this boolean so that saves will work again.
    runAllSteps = False

# Create a button to refresh the cell and another to calculate grade.
grade_button = widgets.Button(description="Calculate Grade")

# Link buttons to functions.
grade_button.on_click(calculate_grade)

# Output area.
gradeOutput = widgets.Output()

# Display the buttons and output.
display(grade_button, gradeOutput)

Button(description='Calculate Grade', style=ButtonStyle())

Output()

### Stopping the Lab

Once you are done with the lab, click on the "Stop Lab" button below. <strong>This will delete your activation, which will delete all of the lab's resources.</strong> Your progress is saved automatically in ```saves/``` within the sidebar of your XDC. You may load this lab in the future by clicking "Load Lab" at the top.

In [10]:
# Click the button below to stop the experiment.
def stoplab(button):
    stop_lab(labname, confirm, stop_output)

# Creating the button.
stopButton = widgets.Button(description="Stop Lab")

# Create a confirmation check.
confirm = widgets.Checkbox(
    value=False,
    description='Confirm',
    disabled=False,
    indent=False
)

# Creating an output area.
stop_output = widgets.Output()

# Run the command on click.
stopButton.on_click(stoplab)

# Display the output.
display(confirm, stopButton, stop_output)

Checkbox(value=False, description='Confirm', indent=False)

Button(description='Stop Lab', style=ButtonStyle())

Output()